# RSNA — baseline quatre lecteurs sur GPU (KAGGLE)

Lire `GUIDE_KAGGLE_FR.md` avant de commencer. Les cinq folds, les images et les
poids sont déjà dans le paquet. Les 513 études CoAtNet locales sont réutilisées.
Ce notebook est livré **non exécuté**. Commencer par `ACTION = "benchmark"`, puis
choisir `"run"` pour entraîner les vingt têtes sur vingt époques, par sessions.

Aucun achat ni contournement de quota : le budget limite notre propre session.
Les encodeurs restent gelés ; les features sont calculées en float32 sur CUDA.

## 1. Vos paramètres

**Prérequis :** GPU NVIDIA activé et paquet joint. **Sortie :** paramètres en mémoire.
`SESSION_HOURS` doit rester inférieur au quota et à la limite de session affichés.
Le mode `benchmark` exécute un petit lot, pas le parcours complet.

In [ ]:
ACTION = "benchmark"            # "benchmark" ou "run"
SESSION_HOURS = 6                 # Arrêt propre puis reprise dans une autre session
READERS = ["coatnet", "small", "tiny", "dino"]
BUNDLE_DIR = ""                   # Automatique : votre Dataset privé dans /kaggle/input
RESUME_FILE = ""                  # Automatique si une seule ancienne sortie est attachée
SESSION_DIR = "/kaggle/temp/rsna_session"
RESULTS_DIR = "/kaggle/working"    # Seules les archives/rapports y seront sauvegardés

## Installation hors ligne et décompression locale

**Prérequis :** paquet disponible et GPU activé. **Sortie :** code, deux dépendances,
JPEG et poids temporaires ; aucune extraction complète ni entraînement à cette étape.
La copie locale des archives peut prendre quelques minutes.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activer le GPU NVIDIA dans les paramètres du notebook avant de continuer.")
if sys.version_info < (3, 11):
    raise RuntimeError("Utiliser une image Python 3.11 ou plus récente.")
print("GPU détecté :", torch.cuda.get_device_name(0))


def extract_zip_safe(archive_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for item in archive.infolist():
            target = (destination / item.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Chemin ZIP interdit.")
        archive.extractall(destination)

if BUNDLE_DIR:
    bundle = Path(BUNDLE_DIR)
else:
    markers = list(Path("/kaggle/input").rglob("bundle_manifest.json"))
    if len(markers) != 1:
        raise FileNotFoundError("Ajouter votre Dataset privé dans Add Input ; renseigner BUNDLE_DIR si plusieurs paquets.")
    bundle = markers[0].parent

manifest = json.loads((bundle / "bundle_manifest.json").read_text())
code_archive = bundle / "code_payload.bin"
with code_archive.open("rb") as stream:
    code_sha = hashlib.file_digest(stream, "sha256").hexdigest()
if code_sha != manifest["files"]["code_payload.bin"]:
    raise ValueError("Archive du code modifiée ou téléversement incomplet.")
runtime = Path(SESSION_DIR) / "runtime"
extract_zip_safe(code_archive, runtime)
sys.path.insert(0, str(runtime))
import bootstrap
resume = bootstrap.find_resume("/kaggle/input", RESUME_FILE)
installation = bootstrap.setup(bundle, SESSION_DIR, RESULTS_DIR, resume=resume)
print(installation)
from knee_baseline.cloud import preflight
preflight()

## Benchmark ou calcul, selon ACTION

`benchmark` mesure seize images pour chaque lecteur. Lire l'estimation avant de
choisir `run`. En `run`, le GPU effectue l'extraction et l'apprentissage des têtes.
L'arrêt sur budget sauvegarde la progression. Pour Kaggle, utiliser Save Version
→ Save & Run All pour le calcul réel et conserver ses fichiers Output.

In [ ]:
from knee_baseline.cloud import benchmark_gpu, run_session

if ACTION == "benchmark":
    benchmark_gpu(images=16)
elif ACTION == "run":
    completed = run_session(hours=SESSION_HOURS, readers=READERS)
    print("Expérience terminée :", completed)
else:
    raise ValueError("ACTION doit être benchmark ou run.")

## Lire l'état et récupérer les résultats

`rsna_resume.tar` permet de continuer sans recalculer les études terminées.
`rsna_reports.zip` regroupe les rapports et soumissions disponibles. Les fichiers
sont affichés dans Output sur Kaggle, ou dans MyDrive/RSNA/results sur Colab.
Une session partielle ne produit pas encore toutes les métriques.

In [ ]:
import json
from pathlib import Path
status = Path(RESULTS_DIR) / "SESSION_STATUS.json"
if status.exists():
    print(json.dumps(json.loads(status.read_text()), indent=2, ensure_ascii=False))
else:
    print("Pas encore de résultats sauvegardés.")
for filename in ["rsna_resume.tar", "rsna_reports.zip"]:
    path = Path(RESULTS_DIR) / filename
    if path.exists():
        print(filename, f"{path.stat().st_size / 1e6:.1f} Mo")

## Continuer une session interrompue

Kaggle : conserver la version terminée, puis attacher ses outputs au notebook
suivant via Add Input (ou téléverser rsna_resume.tar dans un Dataset privé).
Colab : relancer avec le même dossier Drive. Une seule reprise est détectée
automatiquement ; sinon remplir RESUME_FILE. Garder la même image logicielle.

Les trois exemples test vérifient le format ; ils ne mesurent aucune performance
leaderboard. Ne pas ajuster les poids des mélanges sur les mêmes gold OOF.